# 03 - Integracja z MLOps (MLflow)

Ten notebook sluzy do konfiguracji i testu polaczenia z MLflow oraz logowania treningu.

In [ ]:
from pathlib import Path
import time
import numpy as np
import mlflow
import torch

PROJECT_ROOT = Path.cwd().resolve()
import sys
if not (PROJECT_ROOT / 'src').exists():
    p = PROJECT_ROOT
    while True:
        if (p / 'src').exists():
            PROJECT_ROOT = p
            break
        if p == p.parent:
            break
        p = p.parent
sys.path.insert(0, str(PROJECT_ROOT))

MLFLOW_DB = (PROJECT_ROOT / 'mlruns' / 'mlflow.db').resolve()
MLFLOW_TRACKING_URI = f'sqlite:///{MLFLOW_DB}'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

TRAIN_DIR = PROJECT_ROOT / 'data' / 'segmentation' / 'train'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

from src.models.segmentation_detector import SegmentationDetector, TrainConfig
from src.utils.mlflow_logger import MLflowLogger

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('MLflow tracking URI:', MLFLOW_TRACKING_URI)
print('MLflow DB exists:', MLFLOW_DB.exists())
print('Device:', DEVICE)

In [ ]:
with MLflowLogger(experiment_name='segmentation_training', run_name='segmentation_bootstrap') as logger:
    logger.log_params({
        'source_notebook': '03_integracja_mlops',
        'device': DEVICE,
        'train_dir': str(TRAIN_DIR),
    })
    logger.log_metric('bootstrap_ok', 1.0)

print('MLOps bootstrap run zapisany.')

In [ ]:
model_out = MODELS_DIR / 'segmentation_unet_combined.pt'
TRAIN_EPOCHS = 50
TRAIN_BATCH_SIZE = 8
TRAIN_LR = 1e-3
TRAIN_VAL_SPLIT = 0.1
TRAIN_SEED = 42

with MLflowLogger(experiment_name='segmentation_training', run_name='segmentation_train_mlops') as logger:
    logger.log_params({
        'model_out': str(model_out),
        'dataset_dir': str(TRAIN_DIR),
        'epochs': TRAIN_EPOCHS,
        'batch_size': TRAIN_BATCH_SIZE,
        'lr': TRAIN_LR,
        'val_split': TRAIN_VAL_SPLIT,
        'seed': TRAIN_SEED,
        'device': DEVICE,
    })

    if model_out.exists():
        print('Model already exists at', model_out, '- skipping training')
        logger.log_metric('training_skipped_existing_model', 1.0)
    else:
        cfg = TrainConfig(
            dataset_dir=TRAIN_DIR,
            output_path=model_out,
            epochs=TRAIN_EPOCHS,
            batch_size=TRAIN_BATCH_SIZE,
            lr=TRAIN_LR,
            val_split=TRAIN_VAL_SPLIT,
            seed=TRAIN_SEED,
            device=DEVICE,
        )
        detector = SegmentationDetector(device=DEVICE)
        t0 = time.time()
        metrics = detector.train(cfg)
        t1 = time.time()

        logger.log_metrics({
            'train_loss': float(metrics.get('train_loss', np.nan)),
            'val_loss': float(metrics.get('val_loss', np.nan)),
            'best_val_loss': float(metrics.get('best_val_loss', np.nan)),
            'train_time_sec': float(t1 - t0),
        })
        print('Metrics:', metrics)